In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
# Step 1: Fetch JSON Data from an API

import requests

# Fetch data from a public API
url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url)

# Convert response to json
json_data = response.json()
print(json_data)

[{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}, {'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'address': {'street': 'Victor Plains', 'suite': 'Suite 879', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'geo': {'lat': '-43.9509', 'lng': '-34.4618'}}, 'phone': '010-692-6593 x09125', 'website': 'anastasia.net', 'company': {'name': 'Deckow-Crist', 'catchPhrase': 'Proactive didactic contingency', 'bs': 'synergize scalable supply-chains'}}, {'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'address': {'street': 'Douglas Exten

In [4]:
from pprint import pprint
pprint(json_data)

[{'address': {'city': 'Gwenborough',
              'geo': {'lat': '-37.3159', 'lng': '81.1496'},
              'street': 'Kulas Light',
              'suite': 'Apt. 556',
              'zipcode': '92998-3874'},
  'company': {'bs': 'harness real-time e-markets',
              'catchPhrase': 'Multi-layered client-server neural-net',
              'name': 'Romaguera-Crona'},
  'email': 'Sincere@april.biz',
  'id': 1,
  'name': 'Leanne Graham',
  'phone': '1-770-736-8031 x56442',
  'username': 'Bret',
  'website': 'hildegard.org'},
 {'address': {'city': 'Wisokyburgh',
              'geo': {'lat': '-43.9509', 'lng': '-34.4618'},
              'street': 'Victor Plains',
              'suite': 'Suite 879',
              'zipcode': '90566-7771'},
  'company': {'bs': 'synergize scalable supply-chains',
              'catchPhrase': 'Proactive didactic contingency',
              'name': 'Deckow-Crist'},
  'email': 'Shanna@melissa.tv',
  'id': 2,
  'name': 'Ervin Howell',
  'phone': '010-692-6593

In [5]:
# Step 2: Process JSON with PySpark
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import from_json, col, explode, to_json

schema = StructType([
        StructField('id', IntegerType(), True),
        StructField('name', StringType(), True),
        StructField('username', StringType(), True),
        StructField('email', StringType(), True),
        StructField('phone', StringType(), True)
])

df = spark.createDataFrame([(json_data,)], ["json_data"])
df.printSchema()


root
 |-- json_data: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: long (valueContainsNull = true)



In [6]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define schema for the 'geo' object
geo_schema = StructType([
    StructField("lat", StringType(), True),
    StructField("lng", StringType(), True)
])

# Define schema for the 'address' object
address_schema = StructType([
    StructField("street", StringType(), True),
    StructField("suite", StringType(), True),
    StructField("city", StringType(), True),
    StructField("zipcode", StringType(), True),
    StructField("geo", geo_schema, True)
])

# Define schema for the 'company' object
company_schema = StructType([
    StructField("name", StringType(), True),
    StructField("catchPhrase", StringType(), True),
    StructField("bs", StringType(), True)
])

# Define the final top-level schema
user_data_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("username", StringType(), True),
    StructField("email", StringType(), True),
    StructField("address", address_schema, True),
    StructField("phone", StringType(), True),
    StructField("website", StringType(), True),
    StructField("company", company_schema, True)
])

print(type(json_data[0]))

<class 'dict'>


In [7]:
# Assuming 'data' is your list of dictionaries
df = spark.createDataFrame(json_data, schema=user_data_schema)
df.printSchema()
df = df.select('id', 'name', 'username', 'email', 'phone')
df.show(truncate = False)

/# Display the applied schema hierarchy
# df.printSchema()
# df.show()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- suite: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zipcode: string (nullable = true)
 |    |-- geo: struct (nullable = true)
 |    |    |-- lat: string (nullable = true)
 |    |    |-- lng: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- website: string (nullable = true)
 |-- company: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- catchPhrase: string (nullable = true)
 |    |-- bs: string (nullable = true)

+---+------------------------+----------------+-------------------------+---------------------+
|id |name                    |username        |email                    |phone                |
+---+------------------------+----------------+-------------------------

In [17]:
df = spark.createDataFrame(json_data, schema=user_data_schema)
df = df.select('id', 'name', 'username', 'email', 'phone', 'address.city')
df.show(truncate = False)

+---+------------------------+----------------+-------------------------+---------------------+--------------+
|id |name                    |username        |email                    |phone                |city          |
+---+------------------------+----------------+-------------------------+---------------------+--------------+
|1  |Leanne Graham           |Bret            |Sincere@april.biz        |1-770-736-8031 x56442|Gwenborough   |
|2  |Ervin Howell            |Antonette       |Shanna@melissa.tv        |010-692-6593 x09125  |Wisokyburgh   |
|3  |Clementine Bauch        |Samantha        |Nathan@yesenia.net       |1-463-123-4447       |McKenziehaven |
|4  |Patricia Lebsack        |Karianne        |Julianne.OConner@kory.org|493-170-9623 x156    |South Elvis   |
|5  |Chelsey Dietrich        |Kamren          |Lucio_Hettinger@annie.ca |(254)954-1289        |Roscoeview    |
|6  |Mrs. Dennis Schulist    |Leopoldo_Corkery|Karley_Dach@jasper.info  |1-477-935-8478 x6430 |South Christy |
|